# 02 — Agent Definition (AI Engineer owns)

Assembles the single LLM + four tools into a LangGraph ReAct agent. The LLM reasons between every tool call; nothing is hard coded into a fixed sequence.

**Tools:** `eligibility_screener`, `practice_matcher`, `payment_estimator`, `deadline_lookup`. _Only `eligibility_screener` is bound so far (see `agent/graph.py` `TOOLS`); the other three are stubs being built separately._

Out of scope handling is not a tool. The system prompt (PTCF form, written for internal advisors) makes the agent decline irrelevant input and redirect, and drives a short elicitation flow that gathers the client's profile (state and county, acreage, current practices, primary resource concern) across turns.

## Setup

In [1]:
%load_ext autoreload
%autoreload 2

from nrcs_navigator import config
from nrcs_navigator.agent.graph import build_agent

# Prereqs: run notebook 01 first (payment_rates + eCFR vector store populated),
# the Postgres container is up, and OPENAI_API_KEY is set in .env -- both the
# premier model and the eligibility_screener query embedding use it.
print("imports ready")

imports ready


## Build the agent

In [2]:
# Build the ReAct agent on the premier model. model_name comes from config/.env;
# build_agent(config.CHEAP_MODEL) would build the same agent on the cheaper leg.
# Only eligibility_screener is bound right now (see agent/graph.py TOOLS).
agent = build_agent(config.PREMIER_MODEL)
print(f"agent built on {config.PREMIER_MODEL}")

agent built on gpt-4o


## Run an example query

In [3]:
# A realistic in-scope question from an advisor about a client. The agent reasons
# (ReAct), should call eligibility_screener, and answer with citations. thread_id
# keys this conversation in the Postgres checkpointer.
result = agent.invoke(
    {"messages": [("user",
        "I have a client with about 400 acres of cropland in Fremont County, "
        "Iowa. Their main resource concern is soil erosion. Which NRCS programs "
        "might they qualify for?")]},
    config={"configurable": {"thread_id": "demo-eligibility"}},
)

# Full ReAct trace: the question, any tool calls + tool output, then the answer.
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

I have a client with about 400 acres of cropland in Fremont County, Iowa. Their main resource concern is soil erosion. Which NRCS programs might they qualify for?
================================== Ai Message ==================================
Tool Calls:
  eligibility_screener (call_7LuGdw31bEWxpQSzVcUWiMBN)
 Call ID: call_7LuGdw31bEWxpQSzVcUWiMBN
  Args:
    query: 400 acres of cropland in Fremont County, Iowa with a resource concern of soil erosion
================================= Tool Message =================================
Name: eligibility_screener

[7 CFR 1468.3] § 1468.3 Definitions. (ACEP)
Eligible entity means an Indian Tribe, State government, local government, or a nongovernmental organization that has a farmland or grassland protection program that purchases agricultural land easements for the purposes of protecting:
(1) The agricultural use and future viability, and related conservation v

## Demonstrate graceful rejection

In [4]:
# An out-of-scope request: CRP is administered by FSA, not NRCS. The scope guard
# in the system prompt should make the agent decline and redirect to the local
# FSA office WITHOUT calling any tool.
result = agent.invoke(
    {"messages": [("user",
        "Can you help my client enroll in the Conservation Reserve Program (CRP)?")]},
    config={"configurable": {"thread_id": "demo-crp"}},
)
result["messages"][-1].pretty_print()

================================== Ai Message ==================================

The Conservation Reserve Program (CRP) is administered by the Farm Service Agency (FSA), not the NRCS. Your client should contact their local FSA office for assistance with CRP enrollment.
